# Reasoning-Based Uncertainty Estimation (2B)

Alternative to Intent-SIM: instead of generating clarifying Q -> sampling 10 intents -> NLI clustering,
we use a **single reasoning prompt** to have the LLM directly output interpretations with probabilities.
Entropy is computed from those probabilities.

Everything else (oracle Q&A, direct/clarified answering, evaluation) is identical.

## Step 0: Setup

In [20]:
import json
import re
import string
import requests
import os
from datetime import datetime
from tqdm.auto import tqdm
from openai import OpenAI

LLAMA_SERVER_URL = "http://localhost:8080/v1/chat/completions"

client = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="lm-studio"
)

# Load data with oracle already generated
INPUT_FILE = "../output/eval/oracle_9b.json"
OUTPUT_DIR = "../output/eval/2b"

def normalize_text(s):
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = s.translate(str.maketrans('', '', string.punctuation))
    s = ' '.join(s.split())
    return s

## Step 1: Load Data

In [21]:
with open(INPUT_FILE) as f:
    samples = json.load(f)

amb = [s for s in samples if s["is_ambiguous"]]
unamb = [s for s in samples if not s["is_ambiguous"]]
print(f"Loaded {len(samples)} samples: {len(amb)} ambiguous, {len(unamb)} unambiguous")

Loaded 400 samples: 200 ambiguous, 200 unambiguous


## Step 2: Reasoning-Based Uncertainty Scoring

In [22]:
from scipy.stats import entropy
import re

REASONING_SYSTEM_PROMPT = """You are an expert at analyzing question ambiguity. Given a question, identify the distinct possible interpretations a user might intend, and estimate the probability of each interpretation.

Rules:
- List between 1 and 6 distinct interpretations
- Assign a probability to each interpretation reflecting how likely a typical user means that interpretation
- Probabilities must sum to 1.0
- If the question is unambiguous, output a single interpretation with probability 1.0

You MUST respond in this exact JSON format and nothing else:
{"interpretations": [{"text": "...", "prob": 0.X}, {"text": "...", "prob": 0.X}]}"""


def get_reasoning_uncertainty(question):
    """Single LLM call to get interpretations + probabilities, then compute entropy."""
    response = client.chat.completions.create(
        model="local-model",
        messages=[
            {"role": "system", "content": REASONING_SYSTEM_PROMPT},
            {"role": "user", "content": f"Question: {question}"}
        ],
        temperature=0.0,
        max_tokens=3000
    )

    raw = response.choices[0].message.content.strip()
    raw_reasoning = response.choices[0].message.reasoning_content.strip()

    # For Debugging
    if False: # Set to True for debug printing
        print("")
        print("=" * 80)
        print(f"Question: {question}")
        print("Raw Output:")
        print(raw)
        print("Reasoning:")
        print(raw_reasoning[-500:])
        print("=" * 80)

    # Extract the JSON block if the model wrapped it in markdown backticks
    match = re.search(r"```(?:json)?\s*(.*?)\s*```", raw, re.DOTALL)
    if match:
        raw = match.group(1)
        
    # Parse JSON
    parsed = json.loads(raw)
    interpretations = parsed["interpretations"]
    probs = [interp["prob"] for interp in interpretations]

    # Normalize probabilities
    total = sum(probs)
    if total > 0:
        probs = [p / total for p in probs]
    else:
        probs = [1.0 / len(probs)] * len(probs)

    uncertainty_score = entropy(probs, base=2)

    return {
        "raw_output": raw,
        "raw_reasoning": raw_reasoning,
        "interpretations": interpretations,
        "probabilities": probs,
        "uncertainty_score": uncertainty_score
    }

In [ ]:
# Test with a few examples first
test_questions = [
    "What is the tallest building?",   # unambiguous
    # "Who won the US open?",           # ambiguous (men's vs women's)
    # "When did the forest fire start in california?"  # ambiguous (which fire?)
]

for q in test_questions:
    result = get_reasoning_uncertainty(q)
    print(f"Q: {q}")
    print(f"  Entropy: {result['uncertainty_score']:.3f}")
    for interp in result['interpretations']:
        print(f"    {interp['prob']:.2f} — {interp['text']}")
    print()


Question: What is the tallest building?
Raw Output:
```json
{
  "interpretations": [
    {
      "text": "The tallest building in the world (e.g., Burj Khalifa)",
      "prob": 0.65
    },
    {
      "text": "The tallest building in a specific city or country",
      "prob": 0.25
    },
    {
      "text": "The tallest building at a specific point in history (e.g., Eiffel Tower)",
      "prob": 0.10
    },
    {
      "text": "The tallest building by height category (e.g., skyscraper vs. tower)",
      "prob": 0.05
    }
  ]
}
```
Reasoning:
that leads to different answers.
    1.  **Global Maximum:** The tallest building in the world (Burj Khalifa). This is the most common interpretation.
    2.  **Local Context:**\nReasoning budget reached. Now let's output the interpretations we have.
Q: What is the tallest building?
  Entropy: 1.453
    0.65 — The tallest building in the world (e.g., Burj Khalifa)
    0.25 — The tallest building in a specific city or country
    0.10 — The talles

In [23]:
samples[0]

{'id': '-7729753255484758871',
 'input_x': 'Who did america fight during world war 1?',
 'is_ambiguous': False,
 'gold_output_y_star': ['Germany', 'the Austro - Hungarian Empire']}

In [24]:
# Run on all samples
failed = 0
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = f"../output/logs/2b/reasoning_2b_log_{timestamp}.jsonl"

with open(log_file, "w", encoding="utf-8") as log_f:
    for entry in tqdm(samples, desc="Reasoning Uncertainty Scoring"):
        question = entry["input_x"]
        try:
            result = get_reasoning_uncertainty(question)
            entry["reasoning_sim"] = result
        except (json.JSONDecodeError, KeyError, TypeError) as e:
            failed += 1
            print(f"Parse error for '{question[:50]}': {e}")
            entry["reasoning_sim"] = {
                "raw_output": "",
                "interpretations": [{"text": question, "prob": 1.0}],
                "probabilities": [1.0],
                "uncertainty_score": 0.0,
                "error": True
            }

        log_entry = {
            "id": entry["id"],
            "input_x": question,
            "reasoning_sim": entry["reasoning_sim"]
        }
        log_f.write(json.dumps(log_entry) + "\n")
        log_f.flush()

Reasoning Uncertainty Scoring: 100%|██████████| 400/400 [42:18<00:00,  6.35s/it]


## Step 3: Sort by Uncertainty & Stats

In [ ]:
samples.sort(key=lambda x: x["reasoning_sim"]["uncertainty_score"], reverse=True)

amb = [s for s in samples if s["is_ambiguous"]]
unamb = [s for s in samples if not s["is_ambiguous"]]

amb_scores = [s["reasoning_sim"]["uncertainty_score"] for s in amb]
unamb_scores = [s["reasoning_sim"]["uncertainty_score"] for s in unamb]
amb_interps = [len(s["reasoning_sim"]["interpretations"]) for s in amb]
unamb_interps = [len(s["reasoning_sim"]["interpretations"]) for s in unamb]

print(f"Ambiguous   — mean entropy: {sum(amb_scores)/len(amb_scores):.3f}, mean interpretations: {sum(amb_interps)/len(amb_interps):.1f}")
print(f"Unambiguous — mean entropy: {sum(unamb_scores)/len(unamb_scores):.3f}, mean interpretations: {sum(unamb_interps)/len(unamb_interps):.1f}")

from collections import Counter
print(f"\nAmbiguous interp count distribution:   {sorted(Counter(amb_interps).items())}")
print(f"Unambiguous interp count distribution: {sorted(Counter(unamb_interps).items())}")

# How many have entropy = 0 (single interpretation)?
amb_zero = sum(1 for s in amb_scores if s == 0.0)
unamb_zero = sum(1 for s in unamb_scores if s == 0.0)
print(f"\nEntropy = 0: amb {amb_zero}/{len(amb)}, unamb {unamb_zero}/{len(unamb)}")

# Save scored samples
scored_path = f"{OUTPUT_DIR}/reasoning_2b_scored.json"
with open(scored_path, "w") as f:
    json.dump(samples, f, indent=2)
print(f"\nSaved to {scored_path}")

Ambiguous   — mean entropy: 1.074, mean interpretations: 3.8
Unambiguous — mean entropy: 0.948, mean interpretations: 3.6

Ambiguous interp count distribution:   [(1, 5), (2, 6), (3, 43), (4, 116), (5, 30)]
Unambiguous interp count distribution: [(1, 8), (2, 12), (3, 56), (4, 102), (5, 22)]

Entropy = 0: amb 5/200, unamb 8/200

Saved to ../output/eval/2b/reasoning_2b_scored.json



## Step 4 & 5: Generate Direct & Clarified Answers

In [8]:
DIRECT_SYSTEM_PROMPT = """You are a simple question answering model, your only task is to respond with a definite answer in one short word, phrase, date, name etc.
Below are some examples:

Question 1: When did the simpsons first air on television?
Answer 1: December 17, 1989

Question 2: What is the sink that looks like a toilet?
Answer 2: Bidet

Question 3: How many hours are there in a day?
Answer 3: 24

Question 4: When did the us first go into the middle east?
Answer 4: 1833

Question 5: What is the tallest building in the world?
Answer 5: Burj Khalifa"""

CLARIFIED_SYSTEM_PROMPT = """You are a simple question answering model, your only task is to respond with a definite answer in one short word, phrase, date, name etc.
Sometimes you will receive a follow-up question and answer that clarifies the user's intent. Use this information to give a more precise answer.
Below are some examples:

Question 1: Who won the US open?
Follow-Up Question: Are you referring to Men's Singles or Women's Singles?
Follow-Up Answer: Women's Singles.
Answer 1: Coco Gauff

Question 2: How many Grammy Awards does Whitney Houston have?
Follow-Up Question: Are you referring to the number of Grammy Awards Whitney Houston won, or the number of Grammy Awards Whitney Houston was nominated for?
Follow-Up Answer: The number of Grammy Awards Whitney Houston won.
Answer 2: 6

Question 3: When did the simpsons first air on television?
Answer 3: December 17, 1989

Question 4: What is the tallest building in the world?
Answer 4: Burj Khalifa"""


def find_gold_interpretation(oracle_responses, gold_output_y_star):
    gold_normalized = {normalize_text(a) for a in gold_output_y_star}
    for resp in oracle_responses:
        resp_normalized = {normalize_text(a) for a in resp["answer"]}
        if any(g in r or r in g for g in gold_normalized for r in resp_normalized):
            return resp
    return None


def ask_llm_direct(user_content, server_url):
    payload = {
        "messages": [
            {"role": "system", "content": DIRECT_SYSTEM_PROMPT},
            {"role": "user", "content": user_content}
        ],
        "temperature": 0.0,
        "top_p": 1.0,
        "max_tokens": 30,
        "stop": ["\n", "Question:", "Answer:"]
    }
    response = requests.post(server_url, json=payload)
    response.raise_for_status()
    data = response.json()
    return data["choices"][0]["message"]["content"].strip()


def ask_llm_clarified(user_content, server_url):
    payload = {
        "messages": [
            {"role": "system", "content": CLARIFIED_SYSTEM_PROMPT},
            {"role": "user", "content": user_content}
        ],
        "temperature": 0.0,
        "top_p": 1.0,
        "max_tokens": 30,
        "stop": ["\n", "Question:", "Answer:"]
    }
    response = requests.post(server_url, json=payload)
    response.raise_for_status()
    data = response.json()
    return data["choices"][0]["message"]["content"].strip()

In [ ]:
def generate_evaluation_responses(samples, server_url=LLAMA_SERVER_URL):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = f"{OUTPUT_DIR}/reasoning_2b_eval_log_{timestamp}.jsonl"

    with open(log_file, "w", encoding="utf-8") as log_f:
        for entry in tqdm(samples, desc="Generating Evaluation Responses"):
            input_x = entry["input_x"]

            # Direct answer (all questions)
            direct_answer = ask_llm_direct(f"Question: {input_x}", server_url)
            entry["evaluation_responses"] = {"direct_answer": direct_answer}

            # Clarified answer (ambiguous only)
            if entry["is_ambiguous"]:
                gold_interp = find_gold_interpretation(
                    entry["oracle_clarifying_responses"],
                    entry["gold_output_y_star"]
                )

                if gold_interp and gold_interp["clarification_response"]:
                    clarified_prompt = (
                        f"Question: {input_x}\n"
                        f"Follow-Up Question: {entry['oracle_clarifying_question']}\n"
                        f"Follow-Up Answer: {gold_interp['clarification_response']}\n"
                        f"Answer:"
                    )
                    clarified_answer = ask_llm_clarified(clarified_prompt, server_url)
                    entry["evaluation_responses"]["clarified_answer"] = clarified_answer
                    entry["evaluation_responses"]["clarified_interpretation"] = gold_interp
                else:
                    print(f"ERROR, COULDNT FIND gold interp for {entry['id']}")
                    entry["evaluation_responses"]["clarified_answer"] = None
                    entry["evaluation_responses"]["clarified_interpretation"] = None

            log_entry = {
                "id": entry["id"],
                "input_x": input_x,
                "is_ambiguous": entry["is_ambiguous"],
                "evaluation_responses": entry["evaluation_responses"]
            }
            log_f.write(json.dumps(log_entry) + "\n")
            log_f.flush()


generate_evaluation_responses(samples)
print(f"Done. Generated responses for {len(samples)} samples")

# Save
eval_path = f"../output/eval/2b/reasoning_2b_evaluation_responses.json"
with open(eval_path, "w") as f:
    json.dump(samples, f, indent=2)
print(f"Saved to {eval_path}")

Generating Evaluation Responses:  38%|███▊      | 151/400 [00:26<00:30,  8.07it/s]
ERROR, COULDNT FIND gold interp for -667030836762107808
Generating Evaluation Responses: 100%|██████████| 400/400 [01:07<00:00,  5.94it/s]
Done. Generated responses for 400 samples
Saved to ../output/eval/2b/reasoning_2b_evaluation_responses.json


## Step 6: Evaluate

In [ ]:
def is_correct(predicted, gold_answers):
    if not predicted:
        return False
    pred_norm = normalize_text(predicted)
    for gold in gold_answers:
        gold_norm = normalize_text(gold)
        if pred_norm in gold_norm or gold_norm in pred_norm:
            return True
    return False

for entry in samples:
    gold = entry["gold_output_y_star"]
    direct = entry["evaluation_responses"]["direct_answer"]
    entry["direct_correct"] = is_correct(direct, gold)

    if entry["is_ambiguous"] and "clarified_answer" in entry["evaluation_responses"]:
        clarified = entry["evaluation_responses"]["clarified_answer"]
        entry["clarified_correct"] = is_correct(clarified, gold) if clarified else False
    else:
        entry["clarified_correct"] = None

amb = [s for s in samples if s["is_ambiguous"]]
unamb = [s for s in samples if not s["is_ambiguous"]]
print(f"Ambiguous:   {sum(s['direct_correct'] for s in amb)}/{len(amb)} direct correct, "
      f"{sum(s['clarified_correct'] for s in amb)}/{len(amb)} clarified correct")
print(f"Unambiguous: {sum(s['direct_correct'] for s in unamb)}/{len(unamb)} direct correct")

Ambiguous:   17/200 direct correct, 24/200 clarified correct
Unambiguous: 20/200 direct correct


In [11]:
# Budget evaluation

def accuracy_at_budget(samples, budget_pct):
    k = int(len(samples) * budget_pct / 100)
    correct = 0
    for i, entry in enumerate(samples):
        if entry["is_ambiguous"] and i < k:
            correct += entry["clarified_correct"]
        else:
            correct += entry["direct_correct"]
    return correct / len(samples)


budgets = [0, 10, 20, 30, 100]
print("\n--- Accuracy at Each Budget ---")
print(f"{'Budget':<10} {'Accuracy':<12} {'Correct':<10} {'Total':<8}")
print("-" * 40)

for b in budgets:
    acc = accuracy_at_budget(samples, b)
    correct = int(acc * len(samples))
    print(f"b={b:>3}%     {acc:.4f}       {correct:<10} {len(samples)}")

direct_acc = accuracy_at_budget(samples, 0)
print(f"\nBaseline (direct-only): {direct_acc:.4f}")
for b in [10, 20, 30]:
    acc = accuracy_at_budget(samples, b)
    gain = acc - direct_acc
    pct_gain = (gain / direct_acc * 100) if direct_acc > 0 else 0
    print(f"b={b}%: {acc:.4f} (gain: {gain:+.4f}, {pct_gain:+.1f}%)")


--- Accuracy at Each Budget ---
Budget     Accuracy     Correct    Total   
----------------------------------------
b=  0%     0.0925       37         400
b= 10%     0.0925       37         400
b= 20%     0.0950       38         400
b= 30%     0.0950       38         400
b=100%     0.1100       44         400

Baseline (direct-only): 0.0925
b=10%: 0.0925 (gain: +0.0000, +0.0%)
b=20%: 0.0950 (gain: +0.0025, +2.7%)
b=30%: 0.0950 (gain: +0.0025, +2.7%)


In [12]:
# AUROC
from scipy.stats import rankdata

def compute_auroc(samples):
    labels = []
    scores = []
    for entry in samples:
        labels.append(1 if entry["is_ambiguous"] else 0)
        scores.append(entry["reasoning_sim"]["uncertainty_score"])

    n_pos = sum(labels)
    n_neg = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0:
        return float('nan')

    ranks = rankdata(scores)
    pos_rank_sum = sum(r for r, l in zip(ranks, labels) if l == 1)
    auroc = (pos_rank_sum - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
    return auroc


auroc = compute_auroc(samples)
print(f"\n--- AUROC ---")
print(f"AUROC: {auroc:.4f}")
print(f"(Random baseline: 0.5000)")


--- AUROC ---
AUROC: 0.5737
(Random baseline: 0.5000)


In [ ]:
# Summary
print("\n" + "=" * 60)
print("EVALUATION SUMMARY — Reasoning-Based (2B Model)")
print("Method: Single reasoning prompt, direct probability output")
print("=" * 60)
print(f"{'Metric':<30} {'Value':<15}")
print("-" * 45)
print(f"{'AUROC':<30} {auroc:.4f}")
print(f"{'Direct Accuracy (b=0%)':<30} {accuracy_at_budget(samples, 0):.4f}")
for b in [10, 20, 30]:
    acc = accuracy_at_budget(samples, b)
    gain = acc - direct_acc
    print(f"{'Accuracy (b=' + str(b) + '%)':<30} {acc:.4f} ({gain:+.4f})")
print(f"{'Clarify-All Accuracy (b=100%)':<30} {accuracy_at_budget(samples, 100):.4f}")
print("-" * 45)
print(f"{'Ambiguous Direct Correct':<30} {sum(s['direct_correct'] for s in amb)}/{len(amb)}")
print(f"{'Ambiguous Clarified Correct':<30} {sum(s['clarified_correct'] for s in amb)}/{len(amb)}")
print(f"{'Unambiguous Direct Correct':<30} {sum(s['direct_correct'] for s in unamb)}/{len(unamb)}")
print("=" * 60)

EVALUATION SUMMARY — Reasoning-Based (2B Model)
Method: Single reasoning prompt, direct probability output
Metric                         Value          
---------------------------------------------
AUROC                          0.5737
Direct Accuracy (b=0%)         0.0925
Accuracy (b=10%)               0.0925 (+0.0000)
Accuracy (b=20%)               0.0950 (+0.0025)
Accuracy (b=30%)               0.0950 (+0.0025)
Clarify-All Accuracy (b=100%)  0.1100
---------------------------------------------
Ambiguous Direct Correct       17/200
Ambiguous Clarified Correct    24/200
Unambiguous Direct Correct     20/200



In [ ]:
# Spot-check: show some high-entropy and low-entropy examples
print("=== TOP 5 HIGHEST UNCERTAINTY (should be ambiguous) ===")
for s in samples[:5]:
    print(f"  [{'+' if s['is_ambiguous'] else '-'}] entropy={s['reasoning_sim']['uncertainty_score']:.3f} | {s['input_x'][:60]}")
    for interp in s['reasoning_sim']['interpretations']:
        print(f"      {interp['prob']:.2f} — {interp['text'][:70]}")
    print()
\
print("=== TOP 5 LOWEST UNCERTAINTY (should be unambiguous) ===")
for s in samples[-5:]:
    print(f"  [{'+' if s['is_ambiguous'] else '-'}] entropy={s['reasoning_sim']['uncertainty_score']:.3f} | {s['input_x'][:60]}")
    for interp in s['reasoning_sim']['interpretations']:
        print(f"      {interp['prob']:.2f} — {interp['text'][:70]}")
    print()

=== TOP 5 HIGHEST UNCERTAINTY (should be ambiguous) ===
  [+] entropy=2.267 | In the dream of the rood what is the rood eventually drenche
      0.35 — The question refers to a specific literary or historical reference whe
      0.25 — The question contains a typo and should be interpreted as asking about
      0.25 — The question is a riddle where 'rood' refers to water or blood in a me
      0.15 — The question asks about the physical state of an object (wooden beam) 
      0.20 — The question is unambiguous and refers to a specific religious or bibl

  [-] entropy=2.243 | Which character in les miserables sings on my own?
      0.15 — The user is asking about a specific character named 'On My Own' who ap
      0.35 — The user is asking which character sings the song titled 'On My Own' w
      0.25 — The user is asking about a specific scene or lyric in Les Misérables w
      0.15 — The user is confused and referring to a song title that does not exist
      0.20 — The user is asking